# SmartStock AI: Feature Engineering

This notebook develops the M5 preprocessing pipeline incrementally. It begins with one store to keep the later wide-to-long transformation memory-safe.

## Imports and setup

In [213]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEVELOPMENT_STORE_ID, PROCESSED_DATA_DIR
from src.data.load_data import load_raw_data
from src.data.preprocess import select_store_subset

### Select a development store

Use `CA_1` while developing the pipeline. Filtering sales and prices before reshaping reduces the potential long table from about 59 million rows to about 5.9 million rows. The calendar remains complete because it is shared by every store.

In [214]:
raw_data = load_raw_data()
store_data = select_store_subset(raw_data, store_id=DEVELOPMENT_STORE_ID)

calendar_data = store_data["calendar"]
sales_data = store_data["sales"]
sell_price_data = store_data["prices"]

# Release references to the full sales and price tables before reshaping.
del raw_data

day_columns = [column for column in sales_data if column.startswith("d_")]
estimated_long_rows = len(sales_data) * len(day_columns)

print(f"Development store: {DEVELOPMENT_STORE_ID}")
print(f"Calendar shape: {calendar_data.shape}")
print(f"Sales subset shape: {sales_data.shape}")
print(f"Price subset shape: {sell_price_data.shape}")
print(f"Estimated long rows: {estimated_long_rows:,}")

Development store: CA_1
Calendar shape: (1969, 14)
Sales subset shape: (3049, 1947)
Price subset shape: (698412, 4)
Estimated long rows: 5,918,109


### Filter one store

In [215]:
STORE_ID = "CA_1"

store_sales = sales_data[
    sales_data["store_id"] == STORE_ID
].copy()

print("Store:", STORE_ID)
print("Product series:", len(store_sales))
print("Shape:", store_sales.shape)

display(store_sales.head())

Store: CA_1
Product series: 3049
Shape: (3049, 1947)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


In [216]:
print("Unique items:", store_sales["item_id"].nunique())
print("Categories:", store_sales["cat_id"].value_counts())

Unique items: 3049
Categories: cat_id
FOODS        1437
HOUSEHOLD    1047
HOBBIES       565
Name: count, dtype: int64


### Detect daily sales columns

In [217]:
day_cols = [
    col
    for col in store_sales.columns
    if col.startswith("d_")
]

print("Number of day columns:", len(day_cols))
print("First:", day_cols[:5])
print("Last:", day_cols[-5:])

Number of day columns: 1941
First: ['d_1', 'd_2', 'd_3', 'd_4', 'd_5']
Last: ['d_1937', 'd_1938', 'd_1939', 'd_1940', 'd_1941']


### Convert wide to long

In [218]:
id_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
]

sales_long = store_sales.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

print("Long dataset shape:", sales_long.shape)

display(sales_long.head(10))

Long dataset shape: (5918109, 8)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
5,HOBBIES_1_006_CA_1_evaluation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
6,HOBBIES_1_007_CA_1_evaluation,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
7,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,d_1,12
8,HOBBIES_1_009_CA_1_evaluation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2
9,HOBBIES_1_010_CA_1_evaluation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [219]:
expected_rows = len(store_sales) * len(day_cols)

print(f"Expected rows: {expected_rows:,}")
print(f"Actual rows:   {len(sales_long):,}")

assert len(sales_long) == expected_rows

Expected rows: 5,918,109
Actual rows:   5,918,109


### Prepare calendar data

In [220]:
calendar = calendar_data.copy()

calendar["date"] = pd.to_datetime(
    calendar["date"]
)

In [221]:
calendar_duplicates = calendar["d"].duplicated().sum()

print("Duplicate calendar d values:", calendar_duplicates)

assert calendar_duplicates == 0

Duplicate calendar d values: 0


In [222]:
calendar_cols = [
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "d",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI",
]

calendar = calendar[calendar_cols]

### Merge sales + calendar

In [223]:
before_calendar_merge = len(sales_long)

In [224]:
sales_calendar = sales_long.merge(
    calendar,
    on="d",
    how="left",
    validate="many_to_one"
)

In [225]:
after_calendar_merge = len(sales_calendar)

print("Before calendar merge:", f"{before_calendar_merge:,}")
print("After calendar merge: ", f"{after_calendar_merge:,}")

assert before_calendar_merge == after_calendar_merge

Before calendar merge: 5,918,109
After calendar merge:  5,918,109


In [226]:
missing_dates = sales_calendar["date"].isna().sum()

print("Missing dates after calendar merge:", missing_dates)

assert missing_dates == 0

Missing dates after calendar merge: 0


In [227]:
display(
    sales_calendar[
        [
            "item_id",
            "store_id",
            "d",
            "date",
            "sales",
            "weekday",
            "wm_yr_wk"
        ]
    ].head()
)

,item_id,store_id,d,date,sales,weekday,wm_yr_wk
0,HOBBIES_1_001,CA_1,d_1,2011-01-29,0,Saturday,11101
1,HOBBIES_1_002,CA_1,d_1,2011-01-29,0,Saturday,11101
2,HOBBIES_1_003,CA_1,d_1,2011-01-29,0,Saturday,11101
3,HOBBIES_1_004,CA_1,d_1,2011-01-29,0,Saturday,11101
4,HOBBIES_1_005,CA_1,d_1,2011-01-29,0,Saturday,11101


### Prepare price data

In [228]:
store_prices = sell_price_data[
    sell_price_data["store_id"] == STORE_ID
].copy()

print("Price rows:", f"{len(store_prices):,}")

display(store_prices.head())

Price rows: 698,412


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [229]:
price_duplicates = store_prices.duplicated(
    subset=[
        "store_id",
        "item_id",
        "wm_yr_wk"
    ]
).sum()

print("Duplicate price keys:", price_duplicates)

assert price_duplicates == 0

Duplicate price keys: 0


### Merge price data

In [230]:
before_price_merge = len(sales_calendar)

In [231]:
merged_df = sales_calendar.merge(
    store_prices[
        [
            "store_id",
            "item_id",
            "wm_yr_wk",
            "sell_price"
        ]
    ],
    on=[
        "store_id",
        "item_id",
        "wm_yr_wk"
    ],
    how="left",
    validate="many_to_one"
)

In [232]:
after_price_merge = len(merged_df)

print("Before price merge:", f"{before_price_merge:,}")
print("After price merge: ", f"{after_price_merge:,}")

assert before_price_merge == after_price_merge

Before price merge: 5,918,109
After price merge:  5,918,109


In [233]:
print("Final shape:", merged_df.shape)

display(merged_df.head())

Final shape: (5918109, 22)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN


In [234]:
print(merged_df.columns.tolist())

['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price']


### Optimize data types

In [235]:
memory_before = (
    merged_df
    .memory_usage(deep=True)
    .sum()
    / 1024**2
)

print(f"Memory before optimization: {memory_before:.2f} MB")

Memory before optimization: 1437.50 MB


In [236]:
merged_df["sales"] = pd.to_numeric(
    merged_df["sales"],
    downcast="integer"
)

In [237]:
merged_df["sell_price"] = pd.to_numeric(
    merged_df["sell_price"],
    downcast="float"
)

In [238]:
for col in ["wday", "month", "year"]:
    merged_df[col] = pd.to_numeric(
        merged_df[col],
        downcast="integer"
    )

In [239]:
for col in ["snap_CA", "snap_TX", "snap_WI"]:
    if col in merged_df.columns:
        merged_df[col] = pd.to_numeric(
            merged_df[col],
            downcast="integer"
        )

In [240]:
category_cols = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "weekday",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
]

for col in category_cols:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].astype("category")

In [241]:
memory_after = (
    merged_df
    .memory_usage(deep=True)
    .sum()
    / 1024**2
)

print(f"Memory before: {memory_before:.2f} MB")
print(f"Memory after:  {memory_after:.2f} MB")

reduction = (
    (memory_before - memory_after)
    / memory_before
    * 100
)

print(f"Memory reduction: {reduction:.2f}%")

Memory before: 1437.50 MB
Memory after:  510.41 MB
Memory reduction: 64.49%


### Check duplicates

In [242]:
duplicate_rows = merged_df.duplicated(
    subset=[
        "store_id",
        "item_id",
        "d"
    ]
).sum()

print("Duplicate observation keys:", duplicate_rows)

Duplicate observation keys: 0


### Missing-value report

In [243]:
missing_count = merged_df.isna().sum()

missing_percentage = (
    merged_df
    .isna()
    .mean()
    .mul(100)
)

missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_percentage
})

missing_report = (
    missing_report[
        missing_report["missing_count"] > 0
    ]
    .sort_values(
        "missing_percentage",
        ascending=False
    )
)

display(missing_report)

,missing_count,missing_percentage
event_name_2,5905913,99.793921
event_type_2,5905913,99.793921
event_name_1,5436367,91.859866
event_type_1,5436367,91.859866
sell_price,1129842,19.091267


### Convert event nulls into meaningful categories

In [244]:
event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
]

for col in event_cols:
    if col in merged_df.columns:

        # category dtype requires adding new category first
        if isinstance(
            merged_df[col].dtype,
            pd.CategoricalDtype
        ):
            merged_df[col] = (
                merged_df[col]
                .cat.add_categories(["NoEvent"])
            )

        merged_df[col] = (
            merged_df[col]
            .fillna("NoEvent")
        )

In [245]:
merged_df["is_event"] = (
    merged_df["event_name_1"] != "NoEvent"
).astype("int8")

### Investigate missing prices

In [246]:
missing_price_count = (
    merged_df["sell_price"].isna().sum()
)

missing_price_pct = (
    merged_df["sell_price"].isna().mean() * 100
)

print(
    f"Missing prices: {missing_price_count:,} "
    f"({missing_price_pct:.2f}%)"
)

Missing prices: 1,129,842 (19.09%)


In [247]:
missing_price_rows = merged_df[
    merged_df["sell_price"].isna()
]

print(
    "Average sales when price is missing:",
    missing_price_rows["sales"].mean()
)

print(
    "Zero-sales percentage when price is missing:",
    (
        missing_price_rows["sales"].eq(0).mean()
        * 100
    )
)

Average sales when price is missing: 0.0
Zero-sales percentage when price is missing: 100.0


In [248]:
display(
    missing_price_rows[
        [
            "item_id",
            "date",
            "sales",
            "sell_price"
        ]
    ].head(20)
)

,item_id,date,sales,sell_price
0,HOBBIES_1_001,2011-01-29,0,NaN
1,HOBBIES_1_002,2011-01-29,0,NaN
2,HOBBIES_1_003,2011-01-29,0,NaN
3,HOBBIES_1_004,2011-01-29,0,NaN
4,HOBBIES_1_005,2011-01-29,0,NaN
5,HOBBIES_1_006,2011-01-29,0,NaN
6,HOBBIES_1_007,2011-01-29,0,NaN
10,HOBBIES_1_011,2011-01-29,0,NaN
12,HOBBIES_1_013,2011-01-29,0,NaN
13,HOBBIES_1_014,2011-01-29,0,NaN


### Check essential columns for missing data

In [249]:
essential_cols = [
    "item_id",
    "store_id",
    "d",
    "date",
    "sales",
    "cat_id",
    "dept_id",
]

for col in essential_cols:

    missing = merged_df[col].isna().sum()

    print(
        f"{col}: {missing:,} missing"
    )

item_id: 0 missing
store_id: 0 missing
d: 0 missing
date: 0 missing
sales: 0 missing
cat_id: 0 missing
dept_id: 0 missing


### Sort the dataset

In [250]:
merged_df = merged_df.sort_values(
    by=[
        "item_id",
        "date"
    ]
).reset_index(drop=True)

### Remove unnecessary columns

In [251]:
drop_cols = [
    "snap_TX",
    "snap_WI",
]

merged_df = merged_df.drop(
    columns=[
        col
        for col in drop_cols
        if col in merged_df.columns
    ]
)

### Save to folder

In [252]:
from src.config import PROCESSED_DATA_DIR

output_file = (
    PROCESSED_DATA_DIR
    / "ca1_sales_merged.parquet"
)

merged_df.to_parquet(
    output_file,
    index=False
)

### Verify the saved file

In [253]:
test_df = pd.read_parquet(
    output_file
)

print("Reloaded shape:", test_df.shape)

display(test_df.head())

Reloaded shape: (5918109, 21)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,sell_price,is_event
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,1,2011,NoEvent,NoEvent,NoEvent,NoEvent,0,2.0,0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,2,1,2011,NoEvent,NoEvent,NoEvent,NoEvent,0,2.0,0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,3,1,2011,NoEvent,NoEvent,NoEvent,NoEvent,0,2.0,0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,4,2,2011,NoEvent,NoEvent,NoEvent,NoEvent,1,2.0,0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,5,2,2011,NoEvent,NoEvent,NoEvent,NoEvent,1,2.0,0


In [254]:
assert len(test_df) == len(merged_df)

assert (
    test_df["item_id"].nunique()
    ==
    merged_df["item_id"].nunique()
)

print("Parquet validation passed.")

Parquet validation passed.


### Load and sort the dataset

In [255]:
import numpy as np
import pandas as pd
from pathlib import Path

input_file = Path("../data/processed/ca1_sales_merged.parquet")

df = pd.read_parquet(input_file)

df["date"] = pd.to_datetime(df["date"])

df = (
    df.sort_values(["item_id", "date"])
      .reset_index(drop=True)
)

print(df.shape)
display(df.head())

(5918109, 21)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,sell_price,is_event
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,1,2011,NoEvent,NoEvent,NoEvent,NoEvent,0,2.0,0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,2,1,2011,NoEvent,NoEvent,NoEvent,NoEvent,0,2.0,0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,3,1,2011,NoEvent,NoEvent,NoEvent,NoEvent,0,2.0,0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,4,2,2011,NoEvent,NoEvent,NoEvent,NoEvent,1,2.0,0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,5,2,2011,NoEvent,NoEvent,NoEvent,NoEvent,1,2.0,0


### Remove pre-release product history

Missing prices occur only before each product's first known selling price, and all of those rows have zero sales. Treat them as unavailable pre-release periods rather than observed zero demand. This filtering must happen before lag, rolling, price-change, and target features are calculated.

In [256]:
df["release_date"] = (
    df["date"]
    .where(df["sell_price"].notna())
    .groupby(df["item_id"], observed=True)
    .transform("min")
)

if df["release_date"].isna().any():
    raise ValueError("At least one product has no known selling price.")

pre_release = df["date"] < df["release_date"]
pre_release_rows = int(pre_release.sum())

df = df.loc[~pre_release].copy()
df["days_since_release"] = (
    df["date"] - df["release_date"]
).dt.days.astype("int16")

assert df["sell_price"].notna().all()
assert df["days_since_release"].ge(0).all()

print(f"Pre-release rows removed: {pre_release_rows:,}")
print(f"Remaining rows: {len(df):,}")
print(f"Missing prices: {df['sell_price'].isna().sum():,}")

Pre-release rows removed: 1,129,842
Remaining rows: 4,788,267
Missing prices: 0


### Basic calendar features

In [257]:
df["day_of_week"] = df["date"].dt.dayofweek.astype("int8")

df["day_of_month"] = df["date"].dt.day.astype("int8")

df["week_of_year"] = (
    df["date"]
    .dt.isocalendar()
    .week
    .astype("int16")
)

df["quarter"] = df["date"].dt.quarter.astype("int8")

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype("int8")

### Event indicator

In [258]:
df["is_event"] = (
    df["event_name_1"].astype(str) != "NoEvent"
).astype("int8")

In [259]:
df["is_event"] = (
    df["event_name_1"].notna()
).astype("int8")

### Create Previously sell by dated features

In [260]:
lag_days = [1, 7, 14, 28]

for lag in lag_days:
    df[f"lag_{lag}"] = (
        df.groupby("item_id", observed=True)["sales"]
          .shift(lag)
    )

In [261]:
example_item = df["item_id"].iloc[0]

display(
    df[df["item_id"] == example_item][
        [
            "date",
            "sales",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28"
        ]
    ].head(35)
)

,date,sales,lag_1,lag_7,lag_14,lag_28
0,2011-01-29,3,NaN,NaN,NaN,NaN
1,2011-01-30,0,3.0,NaN,NaN,NaN
2,2011-01-31,0,0.0,NaN,NaN,NaN
3,2011-02-01,1,0.0,NaN,NaN,NaN
4,2011-02-02,4,1.0,NaN,NaN,NaN
5,2011-02-03,2,4.0,NaN,NaN,NaN
6,2011-02-04,0,2.0,NaN,NaN,NaN
7,2011-02-05,2,0.0,3.0,NaN,NaN
8,2011-02-06,0,2.0,0.0,NaN,NaN
9,2011-02-07,0,0.0,0.0,NaN,NaN


### Rolling mean features

In [262]:
rolling_windows = [7, 14, 28]

for window in rolling_windows:

    df[f"rolling_mean_{window}"] = (
        df.groupby("item_id", observed=True)["sales"]
          .transform(
              lambda x: (
                  x.shift(1)
                   .rolling(
                       window=window,
                       min_periods=window
                   )
                   .mean()
              )
          )
    )

### Rolling standard deviation to check product demand

In [263]:
for window in [7, 28]:

    df[f"rolling_std_{window}"] = (
        df.groupby("item_id", observed=True)["sales"]
          .transform(
              lambda x: (
                  x.shift(1)
                   .rolling(
                       window=window,
                       min_periods=window
                   )
                   .std()
              )
          )
    )

### Recent demand totals

In [264]:
df["sales_sum_7"] = (
    df.groupby("item_id", observed=True)["sales"]
      .transform(
          lambda x: (
              x.shift(1)
               .rolling(
                   window=7,
                   min_periods=7
               )
               .sum()
          )
      )
)

df["sales_sum_28"] = (
    df.groupby("item_id", observed=True)["sales"]
      .transform(
          lambda x: (
              x.shift(1)
               .rolling(
                   window=28,
                   min_periods=28
               )
               .sum()
          )
      )
)

### Zero demand frequency

In [265]:
df["zero_rate_28"] = (
    df.groupby("item_id", observed=True)["sales"]
      .transform(
          lambda x: (
              x.shift(1)
               .eq(0)
               .rolling(
                   window=28,
                   min_periods=28
               )
               .mean()
          )
      )
)

### Price features

In [266]:
df["price_missing"] = (
    df["sell_price"].isna()
).astype("int8")

In [267]:
df["sell_price_ffill"] = (
    df.groupby("item_id", observed=True)["sell_price"]
      .ffill()
)

In [268]:
df["previous_price"] = (
    df.groupby("item_id", observed=True)["sell_price_ffill"]
      .shift(1)
)

In [269]:
df["price_change"] = (
    df["sell_price_ffill"]
    - df["previous_price"]
)

In [270]:
df["price_change_pct"] = (
    df["price_change"]
    / df["previous_price"]
)

In [271]:
df["price_change_pct"] = (
    df["price_change_pct"]
    .replace([np.inf, -np.inf], np.nan)
)

### Time features

In [272]:
df["dow_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["dow_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

In [273]:
df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

### Create the 7 day target

In [274]:
future_sales = []

grouped_sales = df.groupby(
    "item_id",
    observed=True
)["sales"]

for day_ahead in range(1, 8):
    future_sales.append(
        grouped_sales.shift(-day_ahead)
    )

future_sales_df = pd.concat(
    future_sales,
    axis=1
)

In [275]:
df["target_7d"] = future_sales_df.sum(
    axis=1,
    min_count=7
)

In [276]:
example_item = df["item_id"].iloc[0]

example = (
    df[df["item_id"] == example_item]
    .copy()
    .reset_index(drop=True)
)

display(
    example[
        [
            "date",
            "sales",
            "target_7d"
        ]
    ].head(20)
)

,date,sales,target_7d
0,2011-01-29,3,9.0
1,2011-01-30,0,9.0
2,2011-01-31,0,9.0
3,2011-02-01,1,8.0
4,2011-02-02,4,4.0
5,2011-02-03,2,5.0
6,2011-02-04,0,6.0
7,2011-02-05,2,7.0
8,2011-02-06,0,7.0
9,2011-02-07,0,9.0


In [277]:
row_index = 30

current = example.iloc[row_index]

actual_next_7 = (
    example.iloc[
        row_index + 1 :
        row_index + 8
    ]["sales"].sum()
)

print(
    "Stored target:",
    current["target_7d"]
)

print(
    "Manual next 7 days:",
    actual_next_7
)

Stored target: 16.0
Manual next 7 days: 16


In [278]:
assert np.isclose(
    current["target_7d"],
    actual_next_7
)

print("Target validation passed.")

Target validation passed.


### Check feature nulls

In [279]:
feature_cols = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "sales_sum_7",
    "sales_sum_28",
    "zero_rate_28",
    "target_7d"
]

display(
    df[feature_cols]
    .isna()
    .sum()
    .to_frame("missing")
)

,missing
lag_1,3049
lag_7,21343
lag_14,42686
lag_28,85372
rolling_mean_7,21343
rolling_mean_14,42686
rolling_mean_28,85372
rolling_std_7,21343
rolling_std_28,85372
sales_sum_7,21343


### Remove rows without sufficient history/future

In [280]:
required_model_cols = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "sales_sum_7",
    "sales_sum_28",
    "zero_rate_28",
    "target_7d"
]

model_df = df.dropna(
    subset=required_model_cols
).copy()

In [281]:
print("Before:", f"{len(df):,}")
print("After:", f"{len(model_df):,}")

Before: 4,788,267
After: 4,681,552


In [282]:
missing_price_pct = (
    model_df["sell_price_ffill"]
    .isna()
    .mean()
    * 100
)

print(
    f"Missing price after forward fill: "
    f"{missing_price_pct:.2f}%"
)

Missing price after forward fill: 0.00%


### save the completed CA_1 feature table

In [283]:
feature_output = (
    PROCESSED_DATA_DIR
    / "features"
    / "store_id=CA_1"
    / "features.parquet"
)

feature_output.parent.mkdir(parents=True, exist_ok=True)

model_df.to_parquet(feature_output, index=False)

saved_df = pd.read_parquet(feature_output)

assert len(saved_df) == len(model_df)
assert saved_df.duplicated(["store_id", "item_id", "d"]).sum() == 0

print("Saved:", feature_output)
print("Shape:", saved_df.shape)

Saved: D:\Repos\ML-StockPredict\data\processed\features\store_id=CA_1\features.parquet
Shape: (4681552, 50)
